In [1]:
import warnings
import os 
from pathlib import Path

import pandas as pd
from pathlib import Path
import anndata as ad 
import numpy as np 
from dotenv import dotenv_values

warnings.simplefilter(action='ignore', category=FutureWarning)

In [3]:
# Setting up filepaths from config
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

OUTDIR = DATADIR / "publication" / "Supplemental"
os.makedirs(OUTDIR, exist_ok=True)

SC_IN_FP = DATADIR / "9_objects" / "singlecell" / 'hypomap3d_singlecell_control.h5ad'
OBS_EXT_FP = DATADIR / "9_objects" / "singlecell" / 'hypomap3d_singlecell_all_obs_extended.csv'

L2_INHERITANCE_FP = DATADIR / "10_analyses" / "clustering_comparison" / "inheritance_status_l2.csv"

In [4]:
adata = ad.read_h5ad(SC_IN_FP, backed="r")

# Get additional data for ABC mapping

obs = adata.obs.loc[:, ["CellID"] + [c for c in adata.obs.columns if "descr" in c] + [c for c in adata.obs.columns if "abc_like" in c]] 
obs_ext = pd.read_csv(OBS_EXT_FP, low_memory=False)
obs = obs.merge(obs_ext, on="CellID", how="left")

In [5]:
l2_n = pd.read_csv(L2_INHERITANCE_FP)

In [6]:
abc_to_descr_l2 = adata.obs[["L2_abc_like", "L2_descr"]].drop_duplicates().set_index("L2_abc_like").to_dict()["L2_descr"]

In [7]:
l2_n.source_label = l2_n.source_label.map(abc_to_descr_l2)

In [8]:
sankeydf = obs.loc[(~obs.L2_descr.str.contains("NN")), ["L2_descr", "subclass_id_label_propagated"]].copy()

In [9]:


def filter_sankey_links(
    df,
    source_col,
    target_col,
    min_percent=5,
    top_n=None,
    min_cells_per_source_cluster=0
):
    """
    Filter links for Sankey diagram by:
      - removing weak links (< min_percent of source cluster)
      - removing low-abundance source clusters (< min_cells_per_source_cluster total cells)
      - keeping only top_n target links per source (optional)
    
    Returns a DataFrame with: [source_col, target_col, count, percent, source_idx, target_idx]
    """

    # Step 1: Count each link
    grouped = df.groupby([source_col, target_col]).size().reset_index(name='count')

    # Step 2: Add total count per source cluster
    grouped['total'] = grouped.groupby(source_col)['count'].transform('sum')

    # Step 3: Filter by source cluster size
    grouped = grouped[grouped['total'] >= min_cells_per_source_cluster].copy()

    # Step 4: Calculate percentage per link
    grouped['percent'] = 100 * grouped['count'] / grouped['total']

    # Step 5: Filter by percent threshold
    grouped = grouped[grouped['percent'] >= min_percent].copy()

    # Step 6: Sort and apply top_n filter
    grouped = grouped.sort_values([source_col, 'count'], ascending=[True, False])
    if top_n is not None:
        grouped = grouped.groupby(source_col).head(top_n).reset_index(drop=True)

    return grouped


def plot_sankey(
    filtered_df,
    source_col,
    target_col,
    value_col='count',
    title='Sankey Diagram',
    node_color_map=None,  # Dict: node label -> color
    default_node_color='lightgray',
    link_color='gray',
    width=1000,
    height=600
):
    """
    Plot a Sankey diagram with optional per-node coloring.

    Parameters:
        filtered_df (DataFrame): Output of `filter_sankey_links`.
        node_color_map (dict): Optional. Maps node label to hex color.
        default_node_color (str): Color used if node isn't in map.
    """
    import plotly.graph_objects as go

    # Unique nodes from both columns
    all_nodes = pd.unique(filtered_df[[source_col, target_col]].values.ravel()).tolist()
    node_map = {node: i for i, node in enumerate(all_nodes)}

    # Map to numeric indices
    source_idx = filtered_df[source_col].map(node_map)
    target_idx = filtered_df[target_col].map(node_map)
    values = filtered_df[value_col]

    # Assign colors per node
    if node_color_map:
        node_colors = [node_color_map.get(node, default_node_color) for node in all_nodes]
    else:
        node_colors = default_node_color

    # Build figure
    fig = go.Figure(data=[go.Sankey(
        arrangement="snap",
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=all_nodes,
            color=node_colors
        ),
        link=dict(
            source=source_idx,
            target=target_idx,
            value=values,
            color=link_color
        )
    )])

    fig.update_layout(
        title_text=title,
        font_size=12,
        width=width,
        height=height
    )

    #fig.show()
    return fig

In [10]:
status_to_color = {
    "new cluster": "#221f40",  
    "inherited": "#c4c4a5",
    "inherited and deduplicated": "#d39542"  
}

In [11]:
source_col='L2_descr'
target_col='subclass_id_label_propagated'

# Filter
filtered_links = filter_sankey_links(
    sankeydf,
    source_col=source_col,
    target_col=target_col,
    min_percent=3,
    top_n=5,
    min_cells_per_source_cluster=50
)

# Map each L2 label to its inheritance status
l2_to_status = dict(zip(l2_n['source_label'], l2_n['inheritance_status']))

# Then map L2 label to color
l2_to_color = {l2: status_to_color.get(l2_to_status.get(l2, None), '#d3d3d3') for l2 in filtered_links['L2_descr'].unique()}

# Plot
fig = plot_sankey(
    filtered_links,
    source_col=source_col,
    target_col=target_col,
    title=f'{source_col} -> {target_col} Colored by Inheritance Status',
    node_color_map=l2_to_color,
    default_node_color='lightblue',  # For source nodes
    link_color='rgba(0,0,0,0.2)',
    width=1200,
    height=1800
)

fig.write_image(OUTDIR / "l2_inheritance_sankey.pdf")

/tmp/ipykernel_66740/276893522.py:33: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(OUTDIR / "l2_inheritance_sankey.pdf")
